# Makemore — bigram (full working notebook)

Two models, same output distribution, reached two different ways:
1. **Counting** — tally every bigram, normalize into a row-stochastic matrix
2. **Neural net** — same target, but the distribution is *learned* via gradient descent on a 27-dim → 27-dim linear layer + softmax

The punchline: the neural net converges to the same answer as the counting model. They're equivalent models, just fit two different ways.

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

# one-time download; uncomment to fetch
# !wget -q https://raw.githubusercontent.com/karpathy/makemore/master/names.txt

words = open('names.txt', 'r').read().splitlines()
print(f'{len(words)} names loaded')
print('first five:', words[:5])
print('shortest:', min(len(w) for w in words), '| longest:', max(len(w) for w in words))

In [ ]:
chars = sorted(set(''.join(words)))
stoi = {c: i + 1 for i, c in enumerate(chars)}
stoi['.'] = 0
itos = {i: c for c, i in stoi.items()}
V = len(stoi)
print('vocab:', V, itos)

## Part 1 — counting bigram model

Build a (27, 27) count matrix. Add 1 for smoothing. Normalize rows. Done.

In [ ]:
N = torch.zeros((V, V), dtype=torch.int32)
for w in words:
    chs = ['.'] + list(w) + ['.']
    for a, b in zip(chs, chs[1:]):
        N[stoi[a], stoi[b]] += 1
print('total bigrams:', N.sum().item())

In [ ]:
# Visualize the count matrix (Karpathy's famous heatmap)
fig, ax = plt.subplots(figsize=(14, 14))
fig.patch.set_facecolor('#F5EEDC')
ax.imshow(N, cmap='YlOrBr')
ax.set_title('Bigram counts — rows = previous char, cols = next char', fontsize=13, color='#2A221B')
for i in range(V):
    for j in range(V):
        ax.text(j, i, itos[i] + itos[j], ha='center', va='bottom', color='#2A221B', fontsize=7)
        ax.text(j, i, N[i, j].item(), ha='center', va='top', color='#2A221B', fontsize=6)
ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
# Convert counts to probabilities (with add-one smoothing)
P = (N + 1).float()
P /= P.sum(1, keepdim=True)
print('row sums (should all be 1):', P.sum(1)[:5])

In [ ]:
# Sample names from the counting model
g = torch.Generator().manual_seed(2147483647)
for _ in range(10):
    out, ix = [], 0
    while True:
        p = P[ix]
        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        out.append(itos[ix])
        if ix == 0: break
    print(''.join(out))

In [ ]:
# Evaluate the quality of the counting model: average negative log-likelihood
log_likelihood = 0.0
n = 0
for w in words:
    chs = ['.'] + list(w) + ['.']
    for a, b in zip(chs, chs[1:]):
        log_likelihood += torch.log(P[stoi[a], stoi[b]])
        n += 1
nll = -log_likelihood / n
print(f'counting model avg NLL: {nll.item():.4f}')

## Part 2 — neural-net bigram

Same target distribution, but learned by gradient descent.

In [ ]:
# Build training set of (input char, target char) pairs
xs, ys = [], []
for w in words:
    chs = ['.'] + list(w) + ['.']
    for a, b in zip(chs, chs[1:]):
        xs.append(stoi[a])
        ys.append(stoi[b])
xs = torch.tensor(xs)
ys = torch.tensor(ys)
print('num examples:', xs.nelement())

In [ ]:
# Initialize the 'network' — a single (27, 27) weight matrix
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((V, V), generator=g, requires_grad=True)
print('W shape:', W.shape)

In [ ]:
# Training loop
losses = []
for step in range(200):
    xenc   = F.one_hot(xs, num_classes=V).float()       # (N, 27)
    logits = xenc @ W                                    # (N, 27)
    # F.cross_entropy applies softmax internally — do not softmax logits yourself
    loss = F.cross_entropy(logits, ys) + 0.01 * (W ** 2).mean()  # tiny L2 regularization
    W.grad = None
    loss.backward()
    W.data += -50 * W.grad
    losses.append(loss.item())
    if step % 20 == 0:
        print(f'step {step:3d} | loss {loss.item():.4f}')

print(f'\nfinal NN loss: {losses[-1]:.4f}')
print(f'counting NLL:  {nll.item():.4f}  (these should be very close)')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
fig.patch.set_facecolor('#F5EEDC')
ax.set_facecolor('#FFF9E9')
ax.plot(losses, color='#8B5E3C', lw=2, label='NN loss')
ax.axhline(nll.item(), color='#2E4156', lw=1.5, ls='--', label=f'counting NLL ({nll.item():.3f})')
ax.set_title('Neural-net bigram converges to the counting model', fontsize=13, color='#2A221B')
ax.set_xlabel('step', color='#7A6B5D')
ax.set_ylabel('loss (avg NLL)', color='#7A6B5D')
ax.legend(); ax.grid(alpha=0.3, color='#D4C4B0')
for spine in ax.spines.values(): spine.set_color('#D4C4B0')
plt.tight_layout(); plt.show()

In [ ]:
# Confirm equivalence — the NN's learned distribution should match counting's P
P_nn = F.softmax(W, dim=1).detach()
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor('#F5EEDC')
axes[0].imshow(P.cpu(),    cmap='YlOrBr'); axes[0].set_title('Counting P (row-normalized counts)')
axes[1].imshow(P_nn.cpu(), cmap='YlOrBr'); axes[1].set_title('Neural-net softmax(W)')
for a in axes: a.set_axis_off()
plt.tight_layout(); plt.show()

In [ ]:
# Sample names from the neural-net model
g = torch.Generator().manual_seed(2147483647)
for _ in range(10):
    out, ix = [], 0
    while True:
        xenc   = F.one_hot(torch.tensor([ix]), num_classes=V).float()
        logits = xenc @ W
        probs  = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1, generator=g).item()
        out.append(itos[ix])
        if ix == 0: break
    print(''.join(out))

## Done

The samples are bad. They look vaguely name-ish but mostly random. That's because a bigram only knows the *previous* character. Fixing that is the next video.

Ticks for the [[../02-makemore-bigram/index|checklist]]. Then on to the **[[../03-trigram-exercise/index|trigram exercise]]** — your first model that you build mostly on your own.